# Instalacion

Los siguientes comandos instalan a traves de pip3 las dependencias requeridas por el Notebook

In [23]:
!pip3 install pandas
!pip3 install openpyxl

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


# Lectura del Corpus

En esta funcion, se realiza una lectura del corpus, para poder genera la cantidad de frases que contiene el archivo

In [24]:
from collections import defaultdict

def leer_archivo(nombre_archivo):
    with open(nombre_archivo, 'r', encoding='utf8') as archivo:
        # Esta variable contiene la separacion de palabras que se retornaran en la funcion
        palabras = []

        # Variable temporal para procesar cada palabra
        palabra_actual = []

        # Se itera cada linea del archivo del corpus
        for linea_actual in archivo:

            # Se remueven los caracteres vacios al inicio y final de cada linea
            linea_actual = linea_actual.strip()

            # Si la linea contiene un <doc> o esta vacia, esta linea no se procesa y se continua con la iteracion
            if linea_actual.startswith('<doc') or linea_actual == '':
                continue

            # Se separa cada elemento de la linea por el espacio
            columnas = linea_actual.split()

            # Valida que la linea contenga los elementos: palabra, lema, etiqueta y codigo semantico
            if len(columnas) < 3:
                continue

            # Extraemos palabra y etiqueta y se normaliza a minusculas
            palabra = columnas[0].lower()
            etiqueta = columnas[2]

            # Se agrega la palabra y la etiqueta a la variable temporal
            palabra_actual.append((palabra, etiqueta))

            # Si la etiqueta es un punto final (Fp), cerramos la frase
            if etiqueta == 'Fp':
                palabras.append(palabra_actual)
                palabra_actual = []

        # Por si el archivo no termina en punto, guardamos lo último
        if palabra_actual:
            palabras.append(palabra_actual)

    return palabras

palabras = leer_archivo('corpus-tagged.txt')

# Obtencion de etiquetas, emisiones y transiciones

En esta funcion, se obtienen todas las etiquetas, emisiones y transiciones que existen en el corpus

In [25]:
# Funcion para obtener los conteos de etiquetas, emisiones y transiciones
def obtener_conteos():
    # Se inicializa un diccionario para almacenar las etiquetas
    etiquetas = defaultdict(int)
    # Se inicializa un diccionario para almacenar las emisiones
    emisiones = defaultdict(int)
    # Se inicializa un diccionario para almacenar las transcisiones
    transiciones = defaultdict(int)

    # Se iteran las palabras con etiquetas obtenidas del corpus
    for palabra_etiqueta in palabras:

        # Se inicializa una variable auxiliar
        etiqueta_anterior = '<START>'

        for palabra, etiqueta in palabra_etiqueta:
            # Se incrementa en uno el valor de la etiqueta
            etiquetas[etiqueta] += 1
            # Se incrementa en uno el valor de la emision
            emisiones[(palabra, etiqueta)] += 1
            # Se incrementa en uno el valor de la transcision
            transiciones[(etiqueta_anterior, etiqueta)] += 1

            # Se guarda en la variable auxiliar, la nueva etiqueta
            etiqueta_anterior = etiqueta

        # Se incrementa en uno el valor de la transcision anterior
        transiciones[(etiqueta_anterior, '<END>')] += 1

    return etiquetas, emisiones, transiciones

# Se almacenan las probabilidades de etiquetas, emisiones y transiciones
etiquetas, emisiones, transiciones = obtener_conteos()

# Obtencion de las probabilidades de emision

En esta funcion, se obtienen las probabilidades de las emisiones

In [26]:
# Funcion usada para obtener las probabilidades de emision
def obtener_probabilidades_emision():
    # Se inicializa la variable de probabilidades de emision
    probabilidades = {}
    
    # Se iteran las emisiones
    for (palabra, etiqueta), contador in emisiones.items():
        # se calcula la probabilidad de emision para la palabra e etiqueta
        probabilidades[(palabra, etiqueta)] = contador / etiquetas[etiqueta]

    # Se retorna un array con las probabilidades de emision para cada palabre e etiqueta
    return probabilidades

# Se almacena las probabilidades de emision
probabilidades_emision = obtener_probabilidades_emision()

# Obtencion de las probabilidades de transicion

En esta funcion, se obtienen las probabilidades de las transiciones

In [27]:
# Funcion usada para obtener las probabilidades de transicion
def obtener_probabilidades_transicion():
    # Se inicializa la variable de probabilidades de transicion
    probabilidades = {}

    # Se iteran las transiciones entre etiquetas
    for (etiqueta1, etiqueta2), contador in transiciones.items():
        if etiqueta1 == '<START>':
            total = sum(valor for (e1, e2), valor in transiciones.items() if e1 == '<START>')
        else:
            total = etiquetas[etiqueta1]

        # Se calculan las probabilidades de transicion entre etiquetas contiguas
        probabilidades[(etiqueta1, etiqueta2)] = contador / total

    # Se retorna un array con las probabilidades de transicion entre etiquetas
    return probabilidades

# Se almacena en una variable las probabilidades de transicion entre etiquetas
probabilidades_transicion = obtener_probabilidades_transicion()

# Exportar a probabilidades de emision y transicion a Excel

En esta funcion se exportan las probabilidades de emision y transicion del corpus analizado

In [28]:
import pandas as pd

# Funcion que sirve para exportar las probabilidades de emision
def exportarEmisionExcel(): 
    # Se crea un data set de pandas con las probabilidades de emision que contiene las columnas palabra, etiqueta y probabilidad
    emisionExcel = pd.DataFrame([
        (palabra, etiqueta, probabilidad) for (palabra, etiqueta), probabilidad in probabilidades_emision.items()
    ], columns=['palabra', 'etiqueta', 'probabilidad'])

    # Se exporta a Excel
    emisionExcel.to_excel('probabilidades_emision.xlsx', index=False)

# Funcion que sirve para exportar las probabilidades de transicion
def exportarTransicionExcel():
    # Se crea un data set de pandas con las probabilidades de transicion que contiene las columnas etiqueta anterior, etiqueta siguiente y probabilidad
    transicionExcel = pd.DataFrame([
        (etiqueta1, etiqueta2, probabilidad) for (etiqueta1, etiqueta2), probabilidad in probabilidades_transicion.items()
    ], columns=['etiqueta anterior', 'etiqueta siguiente', 'probabilidad'])

    # Se exporta a Excel
    transicionExcel.to_excel('probabilidades_transicion.xlsx', index=False)

# Se ejecutan las funciones para generar los archivos de Excel
exportarEmisionExcel()
exportarTransicionExcel()

# Algoritmo de Viterbi

En esta funcion se desarrolla el algoritmo de Viterbi

In [29]:
# Función que ejecuta el algoritmo de Viterbi para encontrar la secuencia de etiquetas más probables
def algoritmo_viterbi(palabras, etiquetas, probabilidades_transicion, probabilidades_emision):

    # Se define un suavizador para evitar que el algoritmo se rompa
    suavizador = 1e-10

    # Se crea una lista de diccionarios donde viterbi[t][etiqueta] guarda la probabilidad máxima
    viterbi = [{}]
    # Se crea un diccionario para rastrear el camino de etiquetas recorrido más probable
    ruta = {}

    # Se realiza la inicialización de la primera palabra
    for etiqueta in etiquetas:
        # Se calcula P(etiqueta | START) * P(palabra_0 | etiqueta)
        viterbi[0][etiqueta] = probabilidades_transicion.get(('<START>', etiqueta), suavizador) * \
                               probabilidades_emision.get((palabras[0], etiqueta), suavizador)
        # Se inicializa la ruta para cada etiqueta
        ruta[etiqueta] = [etiqueta]

    # Se realiza la induccion para las siguientes palabras
    for palabra in range(1, len(palabras)):
        viterbi.append({}) # Crea un nuevo diccionario para la palabra actual en la secuencia
        nueva_ruta = {}

        for etiqueta in etiquetas:
            # Para la etiqueta actual, busca cuál de las etiquetas anteriores maximiza la probabilidad
            # Se calcula: Prob_anterior * P(transición) * P(emisión)
            probabilidad, estado = max(
                (viterbi[palabra-1][etiqueta_anterior] *
                 probabilidades_transicion.get((etiqueta_anterior, etiqueta), suavizador) *
                 probabilidades_emision.get((palabras[palabra], etiqueta), suavizador), etiqueta_anterior)
                for etiqueta_anterior in etiquetas
            )

            # Se guarda la probabilidad máxima encontrada para esta etiqueta en este paso
            viterbi[palabra][etiqueta] = probabilidad
            # Se actualiza el camino óptimo acumulado hasta esta etiqueta específica
            nueva_ruta[etiqueta] = ruta[estado] + [etiqueta]

        # Se actualiza la ruta global con los nuevos mejores caminos encontrados en este paso
        ruta = nueva_ruta

    # Se realiza la transición final al estado de cierre
    probabilidad, estado = max(
        (viterbi[len(palabras)-1][etiqueta] *
         probabilidades_transicion.get((etiqueta, '<END>'), suavizador), etiqueta)
        for etiqueta in etiquetas
    )

    # Retorna la probabilidad total del camino, la lista de etiquetas final y la matriz completa
    return probabilidad, ruta[estado], viterbi

# Ejecutar el algoritmo de Viterbi

En esta funcion se muestra la probabilidad de Viterbi dada una oracion

In [30]:
# Funcion que sirve para mostrar la probabilidad de Viterbi de una oracion
def mostrar_probabilidad_viterbi(oracion):
    # Se normaliza a minusculas la oracion
    oracion_limpia = [p.lower() for p in oracion]

    # Se obtiene una lista de etiquetas 
    tags = list(etiquetas.keys())

    # Se ejecuta la funcion para ejecutar el algoritmo de Viterbi
    probabilidad, ruta, viterbi = algoritmo_viterbi(oracion_limpia, tags, probabilidades_transicion, probabilidades_emision)

    # Se pinta en pantala la probabilidad de Viterbi
    print('Probabilidad de Viterbi: ', probabilidad)

# Se crea una variable que contiene un array con palabras de una oracion
oracion_test = ['habla', 'con', 'el', 'enfermo', 'grave', 'de', 'trasplantes', '.']

# Se ejecuta la funcion para mostrar la probabilidad de Viterbi de una oracion
mostrar_probabilidad_viterbi(oracion_test)

Probabilidad de Viterbi:  3.8105847907429224e-16


# Parte 2: Etiquetado morfosintáctico de la oración

Se aplica el algoritmo de Viterbi para obtener la mejor secuencia de etiquetas para las oraciones solicitadas.

Se muestra:
1. La matriz de Viterbi con los estados relevantes
2. La ruta óptima (mejor secuencia de etiquetas)
3. La oración etiquetada

In [31]:
def etiquetar_oracion(oracion, nombre_archivo_excel=None):
    """
    Ejecuta el algoritmo de Viterbi sobre una oración y muestra:
    - La oración etiquetada
    - La matriz de Viterbi (solo estados relevantes)
    - Exporta la matriz a Excel si se proporciona un nombre de archivo
    """
    oracion_limpia = [p.lower() for p in oracion]
    tags = list(etiquetas.keys())
    
    probabilidad, ruta, viterbi = algoritmo_viterbi(oracion_limpia, tags, probabilidades_transicion, probabilidades_emision)
    
    # Mostrar la oración etiquetada
    print("=" * 60)
    print("ORACIÓN ETIQUETADA:")
    print("=" * 60)
    for palabra, tag in zip(oracion, ruta):
        print(f"  {palabra:20s} → {tag}")
    print(f"\nProbabilidad de Viterbi: {probabilidad}")
    print("=" * 60)
    
    # Filtrar solo los estados relevantes: etiquetas que tienen probabilidad
    # de emisión no nula para al menos una palabra de la oración
    estados_relevantes = set()
    for palabra in oracion_limpia:
        for tag in tags:
            if probabilidades_emision.get((palabra, tag), 0) > 0:
                estados_relevantes.add(tag)
    # Incluir también las etiquetas de la ruta óptima
    estados_relevantes.update(ruta)
    estados_relevantes = sorted(estados_relevantes)
    
    # Construir la matriz de Viterbi con estados relevantes
    matriz_datos = {}
    for tag in estados_relevantes:
        matriz_datos[tag] = [viterbi[i].get(tag, 0) for i in range(len(oracion_limpia))]
    
    df_viterbi = pd.DataFrame(matriz_datos, index=oracion).T
    df_viterbi.index.name = 'Etiqueta / Palabra'
    
    print("\nMATRIZ DE VITERBI (estados relevantes):")
    print(df_viterbi.to_string())
    
    # Exportar a Excel si se indica
    if nombre_archivo_excel:
        df_viterbi.to_excel(nombre_archivo_excel)
        print(f"\nMatriz exportada a: {nombre_archivo_excel}")
    
    return ruta, probabilidad, df_viterbi

## Etiquetado de la oración 1: «Habla con el enfermo grave de trasplantes.»

Se ejecuta el algoritmo de Viterbi y se exporta la matriz de probabilidades a Excel.

In [32]:
# Oración 1: «Habla con el enfermo grave de trasplantes.»
oracion1 = ['Habla', 'con', 'el', 'enfermo', 'grave', 'de', 'trasplantes', '.']
ruta1, prob1, df_viterbi1 = etiquetar_oracion(oracion1, 'matriz_viterbi_oracion1.xlsx')

ORACIÓN ETIQUETADA:
  Habla                → VMIP3S0
  con                  → SPS00
  el                   → DA0MS0
  enfermo              → NCMS000
  grave                → AQ0CS0
  de                   → SPS00
  trasplantes          → NCMP000
  .                    → Fp

Probabilidad de Viterbi: 3.8105847907429224e-16

MATRIZ DE VITERBI (estados relevantes):
                           Habla           con            el       enfermo         grave            de   trasplantes             .
Etiqueta / Palabra                                                                                                                
AQ0CS0              1.000000e-20  5.308325e-23  2.905477e-18  5.662904e-18  6.807205e-11  6.807205e-31  8.599535e-25  2.284747e-34
AQ0MS0              1.000000e-20  9.386240e-16  2.905477e-18  1.381196e-09  1.214708e-19  6.807205e-31  8.599535e-25  1.676088e-34
DA0MS0              5.617978e-12  4.693120e-15  2.963587e-06  2.963587e-26  2.208560e-20  1.451247e-29  8.771525e

## Etiquetado de la oración 2: «El enfermo grave habla de trasplantes.»

Se ejecuta el algoritmo de Viterbi para la segunda oración solicitada en la Parte 3.

In [33]:
# Oración 2: «El enfermo grave habla de trasplantes.»
oracion2 = ['El', 'enfermo', 'grave', 'habla', 'de', 'trasplantes', '.']
ruta2, prob2, df_viterbi2 = etiquetar_oracion(oracion2, 'matriz_viterbi_oracion2.xlsx')

ORACIÓN ETIQUETADA:
  El                   → DA0MS0
  enfermo              → NCMS000
  grave                → AQ0CS0
  habla                → NCFS000
  de                   → SPS00
  trasplantes          → NCMP000
  .                    → Fp

Probabilidad de Viterbi: 5.355117481945398e-15

MATRIZ DE VITERBI (estados relevantes):
                              El       enfermo         grave         habla            de   trasplantes             .
Etiqueta / Palabra                                                                                                  
AQ0CS0              1.000000e-20  1.073499e-13  1.290420e-06  1.290420e-26  4.332537e-21  1.208516e-23  3.210817e-33
AQ0MS0              1.000000e-20  2.618290e-05  2.302683e-15  1.290420e-26  2.406965e-22  1.208516e-23  2.355451e-33
DA0MS0              5.617978e-02  5.617978e-22  4.186697e-16  2.751083e-25  4.751824e-28  1.232686e-21  3.021289e-33
Fp                  5.617978e-13  5.617978e-22  4.186697e-15  1.237389e-17  6.258109

# Parte 3: Análisis del etiquetador morfosintáctico

## Pregunta 1: ¿Es correcto el etiquetado morfosintáctico de «Habla con el enfermo grave de trasplantes.»?

El etiquetado obtenido es **parcialmente correcto**. Se analiza palabra por palabra:

- **Habla → VMIP3S0** (verbo, indicativo, presente, 3ª persona singular): La palabra «habla» es ambigua en el corpus, donde aparece como sustantivo (NCFS000, «el habla», 1 vez) y como verbo (VMIP3S0, 6 veces). El etiquetador selecciona la forma verbal, lo cual es razonable en este contexto. Sin embargo, si la oración se interpreta como un imperativo («Habla tú con el enfermo...»), la etiqueta correcta sería VMM02S0 (modo imperativo, 2ª persona singular). El HMM no puede asignar esa etiqueta porque la combinación («habla», VMM02S0) no existe en el corpus de entrenamiento, por lo que no tiene probabilidad de emisión para esa forma. Aun así, VMIP3S0 es una aproximación aceptable si se interpreta como «Él/ella habla con el enfermo...».
- **con → SPS00** (preposición): Correcto.
- **el → DA0MS0** (artículo determinado masculino singular): Correcto.
- **enfermo → NCMS000** (sustantivo común masculino singular): En el corpus aparece como sustantivo (4 veces) y como adjetivo (AQ0MS0, 1 vez). Aquí funciona como sustantivo («el enfermo»), por lo que la etiqueta es correcta.
- **grave → AQ0CS0** (adjetivo calificativo): Correcto. Modifica al sustantivo «enfermo».
- **de → SPS00** (preposición): Correcto.
- **trasplantes → NCMP000** (sustantivo común masculino plural): Correcto.
- **. → Fp** (punto final): Correcto.

En resumen, el etiquetado es correcto para la interpretación en 3ª persona del singular («Él/ella habla...»), pero no captura la posible lectura imperativa de «habla».

## Pregunta 2: Resultado de etiquetar «El enfermo grave habla de trasplantes.»

El resultado del etiquetado se muestra en la celda anterior. En esta oración:

- **El → DA0MS0** (artículo determinado): Correcto.
- **enfermo → NCMS000** (sustantivo): Correcto. «El enfermo» funciona como sujeto.
- **grave → AQ0CS0** (adjetivo): Correcto. Modifica a «enfermo».
- **habla → NCFS000** (sustantivo común femenino singular): **Incorrecto**. El etiquetador asigna a «habla» la etiqueta de sustantivo femenino (como en «el habla castellana»), cuando en esta oración claramente funciona como verbo conjugado en 3ª persona del singular («El enfermo grave habla...»). La etiqueta correcta sería VMIP3S0. Este error se debe a que el modelo HMM bigrama solo considera la etiqueta inmediatamente anterior (AQ0CS0, adjetivo) para calcular la transición, y la probabilidad de transición de adjetivo a sustantivo femenino resulta ser mayor que la de adjetivo a verbo. El modelo no puede captar que «habla» en esta posición es el verbo principal de la oración cuyo sujeto es «El enfermo grave».
- **de → SPS00** (preposición): Correcto.
- **trasplantes → NCMP000** (sustantivo): Correcto.
- **. → Fp** (punto final): Correcto.

El etiquetado de esta segunda oración es **incorrecto** en la palabra «habla». Esto demuestra una limitación importante del modelo bigrama: al considerar solo la etiqueta anterior, no puede captar relaciones sintácticas de largo alcance como la relación sujeto-verbo cuando hay modificadores intermedios.

## Pregunta 3: ¿Cuáles son las limitaciones del etiquetador morfosintáctico creado?

1. **Dependencia del corpus de entrenamiento**: El etiquetador solo puede asignar etiquetas que haya observado en el corpus. Si una palabra aparece en el texto a analizar pero no en el corpus, el etiquetador no tiene probabilidades de emisión reales y recurre al suavizado (valor muy pequeño), lo que puede producir resultados incorrectos. Por ejemplo, si «habla» solo aparece como NCFS000 y VMIP3S0 en el corpus, nunca podrá etiquetar el modo imperativo VMM02S0.

2. **Modelo bigrama limitado**: El HMM bigrama solo considera la etiqueta inmediatamente anterior para calcular las transiciones. No tiene en cuenta contextos más amplios. Esto impide capturar dependencias a larga distancia, como se ha demostrado con la oración 2: la relación sujeto-verbo entre «enfermo» y «habla» se pierde porque el modelo solo ve la transición desde «grave» (adjetivo) y asigna erróneamente «habla» como sustantivo.

3. **No considera información morfológica ni léxica avanzada**: El modelo trata cada palabra como un token aislado. No utiliza información sobre prefijos, sufijos, raíces o familias de palabras que podrían ayudar a desambiguar palabras desconocidas.

4. **Problema con palabras desconocidas (OOV)**: Las palabras que no aparecen en el corpus de entrenamiento reciben una probabilidad de emisión mínima (suavizado) para todas las etiquetas, lo que puede llevar a etiquetados arbitrarios.

5. **Tamaño reducido del corpus**: Un corpus pequeño produce estimaciones de probabilidades poco fiables, especialmente para palabras y transiciones poco frecuentes. Esto aumenta el problema de dispersión de datos (*data sparsity*).

6. **No maneja la ambigüedad semántica**: Dos oraciones con la misma estructura superficial pero diferente significado pueden requerir etiquetados distintos que el modelo no puede distinguir.

## Pregunta 4: ¿Qué posibles mejoras se podrían aplicar?

1. **Utilizar un modelo de n-gramas de orden superior** (trigramas o más): Un HMM basado en trigramas consideraría las dos etiquetas anteriores en lugar de una sola, capturando mejor el contexto y reduciendo errores como el observado en la oración 2, donde la secuencia sustantivo-adjetivo-verbo habría sido mejor capturada.

2. **Ampliar el corpus de entrenamiento**: Utilizar un corpus más grande y diverso mejoraría las estimaciones de probabilidades y reduciría el problema de palabras desconocidas.

3. **Aplicar técnicas de suavizado más sofisticadas**: En lugar de usar un valor constante pequeño para las probabilidades desconocidas, se podrían utilizar técnicas como el suavizado de Laplace (add-one), Good-Turing, Kneser-Ney o interpolación lineal, que distribuyen mejor la probabilidad entre eventos no observados.

4. **Incorporar información morfológica**: Utilizar los sufijos y prefijos de las palabras para estimar probabilidades de emisión de palabras desconocidas. Por ejemplo, las palabras terminadas en «-mente» suelen ser adverbios (RG) y las terminadas en «-ción» suelen ser sustantivos femeninos.

5. **Usar modelos de aprendizaje automático más avanzados**: Se podrían emplear modelos como Conditional Random Fields (CRF), redes neuronales recurrentes (LSTM/GRU) o modelos basados en transformers (como BERT), que capturan dependencias más complejas y contexto bidireccional.

6. **Implementar un análisis morfológico previo**: Añadir un módulo que genere las posibles etiquetas candidatas para cada palabra antes de aplicar Viterbi, reduciendo el espacio de búsqueda y mejorando la precisión.